# AutoDL：A′→CMCR seed1337 联合微调（CE + 0.2 Lovasz）

冻结ResNet50编码器和全部BatchNorm，只使用Val选模，不访问Test。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_DIR = Path('/root/autodl-tmp/projects/lunar-linear/LTL-Net')
DATA_ROOT = Path('/root/autodl-tmp/datasets/dataset_v6_random811_overlap40')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs')
CONFIG_PATH = PROJECT_DIR / 'configs/v6_overlap40_joint_finetune_rezero_cmcr_lovasz02_seed1337.json'
# 如果初始权重不在默认位置，只修改下一行。
INIT_CHECKPOINT = Path('/root/autodl-tmp/outputs/result_v6_overlap40_frozen_gated_rezero1337_cmcr1337_batch4_valfg/best_model.pth')

config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['joint_finetune'] is True and config['seed'] == 1337
assert config['channel_mode'] == 'full' and config['automatic_test_evaluation'] is False
for path in (PROJECT_DIR, DATA_ROOT, INIT_CHECKPOINT):
    assert path.exists(), path
commit = subprocess.check_output(['git', '-C', str(PROJECT_DIR.parent), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Git commit:', commit)
print('配置:', CONFIG_PATH)
print('初始权重:', INIT_CHECKPOINT)
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
command = [sys.executable, str(PROJECT_DIR / 'scripts/run_autodl_joint_finetune.py'),
           '--project-dir', str(PROJECT_DIR), '--config', str(CONFIG_PATH),
           '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
           '--init-checkpoint', str(INIT_CHECKPOINT)]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载:', Path(str(result_dir) + '.zip'))